# Stage 04 - Diagnostics And Train-Inner Ablation (No Reselection)

Route: `lst_models` V2
Scope: `validation_only`; official validation contact is `read_frozen_artifacts_only` (the frozen Stage 03 prediction dump is read and summarized; zero new validation fit-predict events)
User-facing notebook: `notebooks/04_diagnostics_ablation_colab.ipynb`

This notebook executes Stage 04 as pre-registered in `docs/protocols/04_diagnostics_ablation_protocol.md` (D3 core frozen 2026-06-09, operational sections completed 2026-06-10). Stage 04 is diagnosis, not selection: it cannot change the Stage 03 outcome, and its findings become limitation text or pre-registered V2.1 items.


## Frozen Protocol Summary

- Diagnostics arm (measure-only): calibration reliability/ECE/Brier, selective risk-coverage and AURC/E-AURC whole curves, robustness slices with leave-one-slice-out concentration flags, and failure tables — all computed from the frozen `03_validation_predictions.csv` dump. No calibrator fitting, no operating point.
- Same-row slice deltas use the deterministic stratified-dummy reconstruction with dual equality gates against the frozen Stage 03 artifacts; on any mismatch the affected deltas are emitted as `not_computed_due_to_baseline_reconstruction_mismatch` (realized and expectation semantics never mix).
- Ablation arm (train-inner only): the four predeclared architectural controls (`tcn_only`, `dlinear_only`, `last_step_mlp`, `last_step_lightgbm_control`) fit on the Stage 02 fold/row contract with hash-verified same-row identity; budget 4 controls x 1 candidate x 3 folds x 2 seeds = 24 <= 32. The frozen primary is never refit; its train-inner evidence is joined read-only from the frozen Stage 02 trial ledger.
- Controls may not become thesis candidates; the summary carries no ranking or promotion column.


## Expected Artifacts

When `RUN_STAGE04=True`, the runner writes:

```text
results/04_diagnostics_ablation/<run_id>/
  04_calibration_summary.csv
  04_reliability_bins.csv
  04_risk_coverage_curve.csv
  04_selective_summary.csv
  04_robustness_slices.csv
  04_failure_slices.csv
  04_ablation_plan_ledger.csv
  04_ablation_trial_ledger.csv
  04_ablation_summary.csv
  04_diagnostics_report.json
  run_manifest.json
  artifact_inventory.csv
```

The durable-save cell uploads the run folder plus `drive_backup_manifest.json` to `My Drive/lst_models/results/04_diagnostics_ablation/<run_id>/`. Per-control recovery checkpoints are mirrored to `My Drive/lst_models/checkpoints/04_diagnostics_ablation/<run_id>/`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import shutil
import subprocess
import sys
import threading
import time
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "f10637b348b11bab88a207e7ee685c3bed028755"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_DRIVE_SYNC = True
RUN_STAGE03_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_STAGE04_CHECKPOINT_MIRROR = True
RUN_STAGE04_DRIVE_BACKUP = True
RUN_STAGE04 = True
RUN_ARTIFACT_DISPLAY = False

# Exact-run resume (protocol section 13): fill ONLY to complete a crashed
# ablation arm. A resumed run never refits a completed control; a fresh
# run keeps this empty.
RESUME_STAGE04_RUN_ID = ""
RESUME_STAGE04_CHECKPOINT_DIR = ""  # defaults to CHECKPOINT_ROOT / RESUME_STAGE04_RUN_ID

STAGE_NAME = "04_diagnostics_ablation"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
OFFICIAL_VALIDATION_CONTACT = "read_frozen_artifacts_only"
OFFICIAL_VALIDATION_FOR_SELECTION = False
NEW_VALIDATION_FIT_PREDICT_EVENTS = 0
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
STAGE03_RUN_ID = "20260610_133305_716174"
SUPERSEDED_STAGE02_RUN_IDS = ["20260609_100637_704705", "20260610_010019_507648"]
FROZEN_SEEDS = [101, 202]
ABLATION_CONTROL_IDS = ["tcn_only", "dlinear_only", "last_step_mlp", "last_step_lightgbm_control"]
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
STAGE03_OUTPUT_DIR = Path("/content/lst_models_results/03_frozen_validation_readout") / STAGE03_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
STAGE03_DRIVE_PATH_PARTS = ["lst_models", "results", "03_frozen_validation_readout", STAGE03_RUN_ID]
OUTPUT_DIR = Path("/content/lst_models_results/04_diagnostics_ablation")
STAGE04_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "04_diagnostics_ablation"]
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/04_diagnostics_ablation")
CHECKPOINT_DRIVE_BASE_PARTS = ["lst_models", "checkpoints", "04_diagnostics_ablation"]
MAX_ABLATION_MATERIALIZED_BYTES = 2_000_000_000

if STAGE02_RUN_ID in SUPERSEDED_STAGE02_RUN_IDS:
    raise ValueError(f"superseded Stage 02 run id must not be pinned: {STAGE02_RUN_ID}")

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "04_diagnostics_ablation.yaml").exists()
        and (path / "docs" / "protocols" / "04_diagnostics_ablation_protocol.md").exists()
        and (path / "notebooks" / "04_diagnostics_ablation_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "diagnostics_ablation.py").exists()
        and (path / "src" / "lst_models" / "diagnostics.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "<" in PROJECT_REPO_COMMIT:
            raise ValueError("fill PROJECT_REPO_COMMIT with the Stage 04 full-bundle commit before bootstrapping")
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "04_diagnostics_ablation.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "04_diagnostics_ablation_protocol.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "04_diagnostics_ablation_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "diagnostics_ablation.py"
DIAGNOSTICS_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "diagnostics.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
REQUIRED_PROJECT_FILES = [STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH, DIAGNOSTICS_MODULE_PATH, RAW_DATA_MANIFEST_PATH]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("Stage 04 bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_BOOTSTRAP_MODE:", PROJECT_BOOTSTRAP_MODE)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("STAGE00_RUN_ID:", STAGE00_RUN_ID)
print("STAGE01_RUN_ID:", STAGE01_RUN_ID)
print("STAGE02_RUN_ID:", STAGE02_RUN_ID)
print("STAGE03_RUN_ID:", STAGE03_RUN_ID)
print("SUPERSEDED_STAGE02_RUN_IDS:", SUPERSEDED_STAGE02_RUN_IDS)
print("FROZEN_SEEDS:", FROZEN_SEEDS)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("RUN_STAGE04:", RUN_STAGE04)
print("RUN_STAGE04_CHECKPOINT_MIRROR:", RUN_STAGE04_CHECKPOINT_MIRROR)
print("RUN_STAGE04_DRIVE_BACKUP:", RUN_STAGE04_DRIVE_BACKUP)


## Config Load, Runtime Injection, And Contract Check

This cell loads the Stage 04 config sidecar, injects the notebook runtime paths and exact upstream run ids BEFORE the contract asserts (runtime paths are part of the executable contract), and checks the frozen blocks. It does not read the dump or fit anything.


In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the Stage 04 config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required Stage 04 file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage04_config = yaml.safe_load(handle)

stage_inputs = stage04_config["inputs"]
stage_outputs = stage04_config["outputs"]
stage_checkpointing = stage04_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_run_id"] = STAGE02_RUN_ID
stage_inputs["stage02_runtime_run_dir"] = str(STAGE02_OUTPUT_DIR)
stage_inputs["stage02_drive_path_parts"] = STAGE02_DRIVE_PATH_PARTS
stage_inputs["stage03_run_id"] = STAGE03_RUN_ID
stage_inputs["stage03_runtime_run_dir"] = str(STAGE03_OUTPUT_DIR)
stage_inputs["stage03_drive_path_parts"] = STAGE03_DRIVE_PATH_PARTS
stage_inputs["superseded_stage02_run_ids"] = SUPERSEDED_STAGE02_RUN_IDS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)
if RESUME_STAGE04_RUN_ID:
    resume_checkpoint_dir = RESUME_STAGE04_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_STAGE04_RUN_ID)
    stage04_config["resume"] = {
        "enabled": True,
        "run_id": RESUME_STAGE04_RUN_ID,
        "checkpoint_dir": resume_checkpoint_dir,
    }
    print("RESUME MODE: continuing run", RESUME_STAGE04_RUN_ID, "from", resume_checkpoint_dir)

assert stage04_config["stage_name"] == STAGE_NAME
assert stage04_config["route"] == ROUTE
assert stage04_config["scope"] == SCOPE
assert stage04_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert stage04_config["official_validation_contact"] == OFFICIAL_VALIDATION_CONTACT
assert stage04_config["official_validation_for_selection"] is OFFICIAL_VALIDATION_FOR_SELECTION
assert int(stage04_config["new_validation_fit_predict_events"]) == NEW_VALIDATION_FIT_PREDICT_EVENTS
assert stage_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert stage_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert stage_inputs["stage02_run_id"] == STAGE02_RUN_ID
assert stage_inputs["stage03_run_id"] == STAGE03_RUN_ID
assert stage_inputs["stage02_run_id"] not in stage_inputs["superseded_stage02_run_ids"]
assert Path(stage_inputs["raw_data_dir"]) == RAW_DATA_DIR
assert Path(stage_outputs["output_dir"]) == OUTPUT_DIR
assert Path(stage_checkpointing["checkpoint_dir"]) == CHECKPOINT_ROOT
assert stage_checkpointing["checkpoint_drive_path_parts"] == CHECKPOINT_DRIVE_BASE_PARTS
diagnostics_block = stage04_config["diagnostics"]
assert diagnostics_block["source"] == "stage03_validation_predictions_only"
assert diagnostics_block["expected_seeds"] == FROZEN_SEEDS
assert int(diagnostics_block["expected_dump_rows"]) == 302128
assert diagnostics_block["calibration"]["no_calibrator_fitting"] is True
assert diagnostics_block["selective"]["no_operating_point"] is True
assert diagnostics_block["baseline_reconstruction"]["on_mismatch"] == "mark_not_computed_keep_frozen_ticker_rows"
ablation_block = stage04_config["ablation"]
assert sorted(ablation_block["controls"]) == sorted(ABLATION_CONTROL_IDS)
assert ablation_block["candidate_input"] == "price_volume_time_w20"
assert ablation_block["seeds"] == FROZEN_SEEDS
planned_rows = len(ablation_block["controls"]) * 1 * int(ablation_block["n_folds"]) * len(ablation_block["seeds"])
assert planned_rows <= int(ablation_block["budget"]["max_ablation_plan_rows"])
assert int(ablation_block["reference_rows"]["expected_row_count"]) == 6
assert stage04_config["lightgbm_training_defaults"]["early_stopping_validation_source"] == "inner_train_chronological_tail"
assert int(stage04_config["lightgbm_training_defaults"]["early_stopping_rounds"]) == 25
torch_defaults = stage04_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert int(torch_defaults["early_stopping_patience"]) == 8
assert int(torch_defaults["epochs"]) == 64
assert stage04_config["forbidden"]["wording"], "forbidden wording list must be present"
expected_output_names = {
    "calibration_summary": "04_calibration_summary.csv",
    "reliability_bins": "04_reliability_bins.csv",
    "risk_coverage_curve": "04_risk_coverage_curve.csv",
    "selective_summary": "04_selective_summary.csv",
    "robustness_slices": "04_robustness_slices.csv",
    "failure_slices": "04_failure_slices.csv",
    "ablation_plan_ledger": "04_ablation_plan_ledger.csv",
    "ablation_trial_ledger": "04_ablation_trial_ledger.csv",
    "ablation_summary": "04_ablation_summary.csv",
    "diagnostics_report": "04_diagnostics_report.json",
    "manifest": "run_manifest.json",
    "artifact_inventory": "artifact_inventory.csv",
}
for output_key, output_name in expected_output_names.items():
    assert stage_outputs[output_key] == output_name

print(json.dumps({
    "stage_name": stage04_config["stage_name"],
    "source_stage03_run_id": stage_inputs["stage03_run_id"],
    "official_validation_contact": stage04_config["official_validation_contact"],
    "new_validation_fit_predict_events": stage04_config["new_validation_fit_predict_events"],
    "planned_ablation_rows": planned_rows,
    "holdout_test_contact": stage04_config["holdout_test_contact"],
}, indent=2))


## Upstream Inputs: Stage 00-03 Run Folders And Raw Data

Stage 04 verifies the full frozen chain before anything runs. If the Colab runtime was restarted, this cell downloads the four exact upstream run folders by Drive path parts (duplicate Drive matches are a hard error) and the raw ticker files by Drive file ID. The Stage 03 folder includes the frozen prediction dump (~44 MB).


In [ ]:
from lst_models.artifacts import require_artifacts

def get_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive access only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def ensure_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false and mimeType = '{folder_mime}'"
    matches = service.files().list(q=query, fields="files(id, name, webViewLink)", pageSize=10).execute().get("files", [])
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"Duplicate Drive folders named {name!r} under parent {parent_id}")
    created = service.files().create(body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id, name, webViewLink").execute()
    return created["id"]

def ensure_drive_path(service, path_parts):
    folder_id = "root"
    for part in path_parts:
        folder_id = ensure_drive_folder(service, folder_id, part)
    return folder_id

def find_drive_children(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false"
    return service.files().list(q=query, fields="files(id, name, mimeType, size, webViewLink)", pageSize=10).execute().get("files", [])

def upload_or_update_drive_file(service, parent_id, local_path, display_name=None):
    from googleapiclient.http import MediaFileUpload
    name = display_name or local_path.name
    matches = find_drive_children(service, parent_id, name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if len(matches) == 0:
        uploaded = service.files().create(body={"name": name, "parents": [parent_id]}, media_body=media, fields="id, name, size, webViewLink").execute()
        action = "uploaded"
    elif len(matches) == 1:
        uploaded = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id, name, size, webViewLink").execute()
        action = "updated"
    else:
        raise RuntimeError(f"Duplicate Drive files named {name!r} under parent {parent_id}")
    print(f"{action}: {name}")
    return dict(uploaded)

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    raw_source = raw_manifest["raw_source"]
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_source["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = stage_inputs["required_stage00_artifacts"]
required_stage01_artifacts = stage_inputs["required_stage01_artifacts"]
required_stage02_artifacts = stage_inputs["required_stage02_artifacts"]
required_stage03_artifacts = stage_inputs["required_stage03_artifacts"]
if RUN_STAGE04:
    service = get_drive_service()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = [raw_manifest["raw_source"]["files"][ticker]["name"] for ticker in raw_manifest["tickers"] if not (RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]).exists()]
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, stage02_missing, "stage02")
    stage03_missing = [name for name in required_stage03_artifacts if not (STAGE03_OUTPUT_DIR / name).exists()]
    if stage03_missing and RUN_STAGE03_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE03_OUTPUT_DIR, STAGE03_DRIVE_PATH_PARTS, stage03_missing, "stage03")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_OUTPUT_DIR, required_stage02_artifacts)
    require_artifacts(STAGE03_OUTPUT_DIR, required_stage03_artifacts)
    print("Stage 00-03 and raw data input checks passed.")
else:
    print("RUN_STAGE04=False; upstream input sync skipped.")


## Pre-Flight Feasibility Estimate (Before Any Fit)

Protocol section 13: estimate the materialized float32 bytes for the capped ablation fold tensors and check the dump is present and plausibly sized BEFORE any control fit. Abort cleanly with zero fits if the estimate exceeds the predeclared cap.


In [ ]:
if RUN_STAGE04:
    candidate_inputs_path = STAGE01_OUTPUT_DIR / "01_candidate_inputs.json"
    dump_path = STAGE03_OUTPUT_DIR / "03_validation_predictions.csv"
    decision_record_path = STAGE03_OUTPUT_DIR / "03_decision_record.json"
    for path in [candidate_inputs_path, dump_path, decision_record_path]:
        require_path(path)
    decision_record_probe = json.loads(decision_record_path.read_text(encoding="utf-8"))
    print("stage03 decision:", decision_record_probe.get("decision"))
    print("stage03 readout_complete:", decision_record_probe.get("readout_complete"))
    dump_bytes = dump_path.stat().st_size
    print(f"frozen prediction dump bytes: {dump_bytes}")
    if dump_bytes < 1_000_000:
        raise RuntimeError(f"dump file looks truncated ({dump_bytes} bytes); re-sync stage03 from Drive")
    candidate_inputs = json.loads(candidate_inputs_path.read_text(encoding="utf-8"))["candidate_inputs"]
    ablation_candidate = stage04_config["ablation"]["candidate_input"]
    matches = [entry for entry in candidate_inputs if str(entry["candidate_id"]) == ablation_candidate]
    if len(matches) != 1:
        raise ValueError(f"expected exactly one 01_candidate_inputs.json entry for {ablation_candidate!r}; found {len(matches)}")
    window_size = int(matches[0]["window_size"])
    n_features = len(matches[0]["feature_columns"])
    caps = stage04_config["ablation"]["hpo_sample_policy"]
    per_row_bytes = window_size * n_features * 4
    per_fold_rows = int(caps["max_train_samples_per_fold"]) + int(caps["max_eval_samples_per_fold"])
    estimate_bytes = per_fold_rows * per_row_bytes * int(stage04_config["ablation"]["n_folds"])
    print(json.dumps({
        "window_size": window_size,
        "n_features": n_features,
        "per_fold_capped_rows": per_fold_rows,
        "estimated_materialized_bytes": estimate_bytes,
        "max_ablation_materialized_bytes": MAX_ABLATION_MATERIALIZED_BYTES,
    }, indent=2))
    if estimate_bytes > MAX_ABLATION_MATERIALIZED_BYTES:
        raise RuntimeError(
            "Stage 04 pre-flight infeasible BEFORE any fit: estimated materialized "
            f"bytes {estimate_bytes} exceed the cap {MAX_ABLATION_MATERIALIZED_BYTES}; zero control fits have occurred"
        )
    print("Pre-flight feasibility passed; proceeding is authorized.")
else:
    print("RUN_STAGE04=False; pre-flight feasibility estimate skipped.")


## Checkpoint Mirror To Drive

The runner writes per-control recovery checkpoints (partial trial ledger plus `checkpoint_manifest.json`) under the local checkpoint root. This cell defines a background mirror that uploads mtime-stable checkpoint files to `My Drive/lst_models/checkpoints/04_diagnostics_ablation/<run_id>/` during the run. Checkpoints are recovery state only, never evidence artifacts.


In [ ]:
STAGE04_MIRROR_STATE = {"stop": False, "errors": [], "uploaded_mtimes": {}, "folder_ids": {}}
STAGE04_MIRROR_POLL_SECONDS = 120
STAGE04_MIRROR_STABLE_SECONDS = 10

def mirror_stage04_checkpoints_once(service):
    if not CHECKPOINT_ROOT.exists():
        return 0
    uploaded_count = 0
    now = time.time()
    for path in sorted(CHECKPOINT_ROOT.rglob("*")):
        if not path.is_file():
            continue
        mtime = path.stat().st_mtime
        if now - mtime < STAGE04_MIRROR_STABLE_SECONDS:
            continue
        key = str(path)
        if STAGE04_MIRROR_STATE["uploaded_mtimes"].get(key) == mtime:
            continue
        relative = path.relative_to(CHECKPOINT_ROOT)
        parent_parts = tuple(CHECKPOINT_DRIVE_BASE_PARTS + list(relative.parent.parts)) if relative.parent != Path(".") else tuple(CHECKPOINT_DRIVE_BASE_PARTS)
        folder_id = STAGE04_MIRROR_STATE["folder_ids"].get(parent_parts)
        if folder_id is None:
            folder_id = ensure_drive_path(service, list(parent_parts))
            STAGE04_MIRROR_STATE["folder_ids"][parent_parts] = folder_id
        upload_or_update_drive_file(service, folder_id, path)
        STAGE04_MIRROR_STATE["uploaded_mtimes"][key] = mtime
        uploaded_count += 1
    return uploaded_count

def stage04_checkpoint_mirror_loop():
    try:
        mirror_service = get_drive_service()
    except Exception as exc:
        STAGE04_MIRROR_STATE["errors"].append(f"mirror auth failed: {type(exc).__name__}: {exc}")
        print("checkpoint mirror disabled:", STAGE04_MIRROR_STATE["errors"][-1])
        return
    while not STAGE04_MIRROR_STATE["stop"]:
        try:
            mirror_stage04_checkpoints_once(mirror_service)
        except Exception as exc:
            STAGE04_MIRROR_STATE["errors"].append(f"{type(exc).__name__}: {exc}")
            print("checkpoint mirror error (recorded, run continues):", STAGE04_MIRROR_STATE["errors"][-1])
        for _ in range(STAGE04_MIRROR_POLL_SECONDS):
            if STAGE04_MIRROR_STATE["stop"]:
                break
            time.sleep(1)

def start_stage04_checkpoint_mirror():
    STAGE04_MIRROR_STATE["stop"] = False
    thread = threading.Thread(target=stage04_checkpoint_mirror_loop, name="stage04_checkpoint_mirror", daemon=True)
    thread.start()
    return thread

RESUME_REQUIRED_CHECKPOINT_FILES = [
    "checkpoint_manifest.json",
    "04_ablation_trial_ledger_partial.csv",
]

def fetch_resume_checkpoint_from_drive():
    """Restore the exact checkpoint folder for RESUME_STAGE04_RUN_ID from
    Drive when the local runtime lost it (exact path parts, no scans)."""
    local_dir = Path(RESUME_STAGE04_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_STAGE04_RUN_ID))
    missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if not missing:
        print("resume checkpoint complete locally:", local_dir)
        return
    service = get_drive_service()
    drive_parts = CHECKPOINT_DRIVE_BASE_PARTS + [RESUME_STAGE04_RUN_ID]
    sync_stage_artifacts_from_drive(service, local_dir, drive_parts, missing, "stage04 resume checkpoint")
    still_missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if still_missing:
        raise FileNotFoundError(f"resume checkpoint files missing after Drive fetch: {still_missing}")

if RUN_STAGE04 and RESUME_STAGE04_RUN_ID:
    fetch_resume_checkpoint_from_drive()

def stop_stage04_checkpoint_mirror(thread):
    STAGE04_MIRROR_STATE["stop"] = True
    if thread is not None:
        thread.join(timeout=STAGE04_MIRROR_POLL_SECONDS + 30)
    final_service = get_drive_service()
    STAGE04_MIRROR_STATE["uploaded_mtimes"] = {}
    uploaded = mirror_stage04_checkpoints_once(final_service)
    print(f"final checkpoint sweep uploaded {uploaded} files; recorded mirror errors: {len(STAGE04_MIRROR_STATE['errors'])}")
    if STAGE04_MIRROR_STATE["errors"]:
        print(json.dumps(STAGE04_MIRROR_STATE["errors"], indent=2))

print("Checkpoint mirror helpers ready; RUN_STAGE04_CHECKPOINT_MIRROR =", RUN_STAGE04_CHECKPOINT_MIRROR)


## Run Stage 04 (Diagnostics + Ablation)

The package-backed entry point is `lst_models.stages.diagnostics_ablation.run_stage`. The diagnostics arm only reads the frozen dump; the ablation arm fits the four predeclared controls on hash-verified Stage 02 train-inner rows. No model is ever fit, predicted, thresholded, or calibrated on official validation rows. The committed notebook remains inert by default.


In [ ]:
if RUN_STAGE04:
    try:
        from lst_models.stages.diagnostics_ablation import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing Stage 04 package entry point: src/lst_models/stages/diagnostics_ablation.py.") from exc
    mirror_thread = start_stage04_checkpoint_mirror() if RUN_STAGE04_CHECKPOINT_MIRROR else None
    try:
        result = run_stage(stage04_config)
    finally:
        if RUN_STAGE04_CHECKPOINT_MIRROR:
            stop_stage04_checkpoint_mirror(mirror_thread)
    display(result)
    STAGE04_RUN_ID = Path(result.run_dir).name
    if Path(result.run_dir).parent != OUTPUT_DIR:
        raise RuntimeError(f"Stage 04 output dir mismatch: {Path(result.run_dir).parent} != {OUTPUT_DIR}")
    print("STAGE04_RUN_ID:", STAGE04_RUN_ID)
else:
    print("RUN_STAGE04=False; Stage 04 not executed.")


## Durable Drive Result Save

Immediately after `run_stage` returns, this cell validates the twelve required Stage 04 outputs (imported from the stage module, never retyped), refuses upload unless the run manifest records `new_validation_fit_predict_events=0`, `official_validation_contact=read_frozen_artifacts_only`, and `holdout_test_contact=false`, and uploads the run folder to `My Drive/lst_models/results/04_diagnostics_ablation/<run_id>/`. `drive_backup_manifest.json` is written and uploaded last; its own entry records size null (self-reference).


In [ ]:
from lst_models.stages.diagnostics_ablation import REQUIRED_STAGE04_ARTIFACTS

def backup_stage04_results_to_drive(output_run_dir):
    run_dir = Path(output_run_dir)
    if not run_dir.exists():
        raise FileNotFoundError(f"Stage 04 output folder not found: {run_dir}")
    run_id = run_dir.name
    missing = [name for name in REQUIRED_STAGE04_ARTIFACTS if not (run_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required Stage 04 artifacts before Drive backup in "
            f"{run_dir}: {missing}"
        )
    run_manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
    if int(run_manifest.get("new_validation_fit_predict_events", -1)) != 0:
        raise ValueError("Stage 04 backup refused: run manifest must record new_validation_fit_predict_events=0")
    if run_manifest.get("official_validation_contact") != "read_frozen_artifacts_only":
        raise ValueError("Stage 04 backup refused: run manifest must record official_validation_contact=read_frozen_artifacts_only")
    if run_manifest.get("holdout_test_contact") is not False:
        raise ValueError("Stage 04 backup refused: run manifest must record holdout_test_contact=false")
    if run_manifest.get("no_final_model_selected") is not True:
        raise ValueError("Stage 04 backup refused: run manifest must record no_final_model_selected=true")
    if run_manifest.get("source_stage03_run_id") != STAGE03_RUN_ID:
        raise ValueError(f"Stage 04 backup refused: manifest source_stage03_run_id != {STAGE03_RUN_ID}")
    diagnostics_report = json.loads((run_dir / "04_diagnostics_report.json").read_text(encoding="utf-8"))
    service = get_drive_service()
    drive_path_parts = STAGE04_DRIVE_RESULT_PATH_PARTS + [run_id]
    drive_folder_id = ensure_drive_path(service, drive_path_parts)
    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    local_files = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.name != backup_manifest_path.name)
    uploads = []
    for path in local_files:
        uploaded = upload_or_update_drive_file(service, drive_folder_id, path)
        uploaded["bytes"] = path.stat().st_size
        uploads.append(uploaded)
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_id,
        "stage_run_id": run_id,
        "baseline_reconstruction_status": diagnostics_report.get("baseline_reconstruction_status"),
        "new_validation_fit_predict_events": run_manifest.get("new_validation_fit_predict_events"),
        "official_validation_rows_read": run_manifest.get("official_validation_rows_read"),
        "local_output_dir": str(run_dir),
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": drive_folder_id,
        "uploaded_file_names": [upload["name"] for upload in uploads],
        "uploaded_files": uploads,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "holdout_test_contact": run_manifest.get("holdout_test_contact"),
        "official_validation_contact": run_manifest.get("official_validation_contact"),
        "official_validation_for_selection": run_manifest.get("official_validation_for_selection"),
        "no_final_model_selected": run_manifest.get("no_final_model_selected"),
    }
    backup_manifest["uploaded_files"] = backup_manifest["uploaded_files"] + [
        {"name": backup_manifest_path.name, "bytes": None, "self_reference": True}
    ]
    backup_manifest["uploaded_file_names"].append(backup_manifest_path.name)
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    print("stage_run_id:", backup_manifest["stage_run_id"])
    print("drive_path:", backup_manifest["drive_path"])
    print("drive_folder_id:", backup_manifest["drive_folder_id"])
    return backup_manifest

if RUN_STAGE04_DRIVE_BACKUP and RUN_STAGE04:
    if "result" not in globals():
        raise RuntimeError("RUN_STAGE04_DRIVE_BACKUP=True requires running Stage 04 first.")
    stage04_drive_backup_manifest = backup_stage04_results_to_drive(result.run_dir)
else:
    print("RUN_STAGE04_DRIVE_BACKUP is disabled or RUN_STAGE04=False; no Stage 04 result backup uploaded.")


## Diagnostics Display

After an approved run, display only Stage 04 artifacts from `result.run_dir`: the diagnostics report summary, the ablation summary against the read-only Stage 02 reference rows, compact calibration and selective tables, and whole-curve plots. Nothing here marks, recommends, or selects an operating point.


In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd
    import matplotlib.pyplot as plt

    if "result" not in globals():
        raise RuntimeError("RUN_ARTIFACT_DISPLAY=True requires running Stage 04 first.")
    run_dir = Path(result.run_dir)
    report = json.loads((run_dir / "04_diagnostics_report.json").read_text(encoding="utf-8"))
    print(json.dumps({
        "stage03_decision": report.get("stage03_decision"),
        "baseline_reconstruction_status": report.get("baseline_reconstruction_status"),
        "concentration_loo_sign_flips": report.get("concentration_loo_sign_flips"),
        "ablation_completed_rows_by_control": report.get("ablation_completed_rows_by_control"),
        "official_validation_rows_read": report.get("official_validation_rows_read"),
        "new_validation_fit_predict_events": report.get("new_validation_fit_predict_events"),
        "deferred_items": report.get("deferred_items"),
    }, indent=2))
    display(pd.read_csv(run_dir / "04_ablation_summary.csv"))
    calibration = pd.read_csv(run_dir / "04_calibration_summary.csv")
    display(calibration.loc[calibration["ticker"].eq("pooled")])
    display(pd.read_csv(run_dir / "04_selective_summary.csv"))
    slices = pd.read_csv(run_dir / "04_robustness_slices.csv")
    display(slices.loc[slices["slice_axis"].isin(["ticker", "seed", "calendar_year"])])

    bins = pd.read_csv(run_dir / "04_reliability_bins.csv")
    primary_bins = bins.loc[
        bins["view"].eq("p_up") & bins["binning_scheme"].eq("equal_width") & bins["n_bins"].eq(15)
    ]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for seed, seed_bins in primary_bins.groupby("seed"):
        axes[0].plot(seed_bins["mean_predicted"], seed_bins["empirical_frequency"], marker="o", label=f"seed {seed}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    axes[0].set_xlabel("mean predicted P(up)")
    axes[0].set_ylabel("empirical up-frequency")
    axes[0].set_title("Reliability (p_up, equal-width 15 bins)")
    axes[0].legend()
    curve = pd.read_csv(run_dir / "04_risk_coverage_curve.csv")
    for seed, seed_curve in curve.groupby("seed"):
        axes[1].plot(seed_curve["coverage"], seed_curve["selective_risk"], label=f"seed {seed}")
    axes[1].set_xlabel("coverage")
    axes[1].set_ylabel("selective risk (1 - accuracy)")
    axes[1].set_title("Risk-coverage (whole curve)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("RUN_ARTIFACT_DISPLAY=False; no Stage 04 artifacts displayed.")


## Honest Interpretation Template

Stage 04 is diagnosis, not selection. Allowed wording: `validation-only evidence`, `official validation readout`, `candidate met/did not meet predeclared validation-readout criteria`.

Do not claim a final model, official validation winner, holdout winner, test winner, proved best model, proven generalization, profitability, holdout-readiness, selection by official validation, or a chosen threshold from this notebook. Ablation controls may not be promoted to thesis candidates; `no_final_model_selected=true` is retained in every Stage 04 output.

Required context sentences for any summary based on this run:

- Stage 04 read and summarized the frozen Stage 03 validation prediction dump (read contact is recorded in the manifest as `official_validation_rows_read`); it produced zero new validation fit-predict events.
- Ablation absolute metrics come from capped train-inner folds and are not comparable to Stage 03 absolute metrics (Zadrozny 2004); only same-row deltas within each evidence tier support claims.
- `activity_tercile` is a dump-native band-pass activity proxy, not realized volatility; a raw-bar volatility slice is a deferred V2.1 item, as is SHAP/permutation importance.
- Boundary calendar years are partial periods; read slice rows with their `n_rows` column.
- If the baseline reconstruction status is `mismatch_deltas_not_computed`, the affected slice deltas and leave-one-out flags are intentionally absent rather than approximated; only the frozen Stage 03 per-ticker deltas remain authoritative.
